In [1]:
from langchain_openai import ChatOpenAI

import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_KEY  = os.getenv("OPENROUTER_API_KEY")

llm = ChatOpenAI(model="stepfun/step-3.5-flash:free", api_key=OPENROUTER_KEY, base_url=OPENROUTER_BASE, max_retries=2)

/Users/bala/python-envs/AI_Agents/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model="qwen/qwen3-embedding-8b", base_url=OPENROUTER_BASE, api_key=OPENROUTER_KEY)

from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    collection_name="pagewise_recursive_chunk_with_chunk_size_2000",
    persist_directory="./chroma_db_3",  
    embedding_function=embedding
)



/var/folders/l5/fhk8809n7g17qfqjg0g269vh0000gn/T/ipykernel_52054/4160562175.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [3]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={'k':25})

In [4]:
def format_docs_with_citations(docs):
    blocks = []
    for i,doc in enumerate(relevant_docs,1):
        info = []
        metadata = doc.metadata
        if metadata['chapter_num']:
            info.append('Chapter '+metadata['chapter_num'])
        if metadata['chapter_title']:
            info.append('Chapter '+ metadata.get('chapter_num',' ')+metadata['chapter_title'])
        if metadata['section_title']:
            info.append('Section '+ metadata.get('section_num',' ')+metadata['section_title'])
        if metadata['article_title']:
            info.append('Article '+ metadata.get('article',' ')+metadata['article_title'])

        label = " | ".join(info)
        blocks.append(
                f"[SOURCE {i}]\n"
                f"Reference : {label}\n"
                f"Page      : {metadata.get('page_num', 'N/A')}\n"
                f"---\n"
                f"{doc.page_content.strip()}"
            )

    blocks = "\n\n".join(blocks)
    return blocks

In [5]:
query = "Can i build my own LLM using the data of my fried's Facebook data?"
relevant_docs = retriever.invoke(query)
blocks = format_docs_with_citations(relevant_docs)

In [6]:
print(blocks)

[SOURCE 1]
Reference : Chapter REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL | Article (Text with EEA relevance)
Page      : 31
---
instruments of Union law in particular having in mind the usage and the perception of their recipients, the AI
systems subject to this Regulation may be provided as intermediary services or parts thereof within the meaning of
Regulation (EU) 2022/2065, which should be interpreted in a technology-neutral manner. For example, AI systems
may be used to provide online search engines, in particular, to the extent that an AI system such as an online chatbot
performs searches of, in principle, all websites, then incorporates the results into its existing knowledge and uses the
updated knowledge to generate a single output that combines different sources of information.
(120)
Furthermore, obligations placed on providers and deployers of certain AI systems in this Regulation to enable the
detection and disclosure that the outputs of those 

In [7]:
system = """ 
        You are a precise legal assistant for the EU AI Act.
        Answer the user's question using ONLY the provided source passages.

        Rules:
        - Cite every claim using [SOURCE N] inline, e.g. "Providers must register ... [SOURCE 2]."
        - After your answer, add a 'References' section listing only the sources you actually used.
        - Each reference line: [SOURCE N] — <Reference label>, Page <page_num>
        - If the answer is not in the sources, say so — do not speculate.
        - Be concise but complete.
    """

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Sources:\n\n{context}\n\nQuestion: {question}"),
])

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context":  retriever | format_docs_with_citations,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
response = rag_chain.invoke("Can i build my own LLM using the data of my fried's Facebook data?")                           

In [11]:
print(response)

Based on the provided passages from the EU AI Act, building your own LLM using your friend's Facebook data raises significant legal concerns under Union data protection law.

The EU AI Act operates within the framework of existing Union law, notably the fundamental right to data protection. The provided text repeatedly references Regulations (EU) 2016/679 (GDPR), (EU) 2018/1725, and Directive (EU) 2016/680 as the foundation for processing personal data [SOURCE 8]. It explicitly states that "the right to privacy and to protection of personal data must be guaranteed throughout the entire lifecycle of the AI system" and that principles like data minimisation and data protection by design must be followed [SOURCE 7].

Using your friend's personal Facebook data for training an AI model would constitute processing of personal data. Such processing requires a lawful basis under the GDPR, most commonly explicit consent from the data subject (your friend). The sources do not provide any excepti

### So it took 25 nearest chunks to get somewhat good result, and we can't say it's 100% perfect yet, 
### so let's try to improve them by iteration over iteration

## One of the Most effective way to improve our result is to use HyDE - Hypothetical Document Embedding

In [12]:
from langchain_core.prompts import ChatPromptTemplate

hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a EU AI Act legal expert. Given a user question, write a short "
     "2-3 sentence passage that sounds like it came directly from the EU AI Act "
     "or its recitals. Use formal legal language about training data, personal "
     "data, copyright, and AI model development. Do not answer the question — "
     "just write the kind of passage that WOULD answer it."),
    ("human", "{question}")
    
])
    

In [13]:
hyde_chain = hyde_prompt | llm | StrOutputParser()

In [14]:
response = hyde_chain.invoke("Can i build my own LLM using the data of my fried's Facebook data?")
print(response)

The processing of personal data, including that extracted from social media platforms such as Facebook, for the purpose of training an AI model is subject to the stringent conditions of Regulation (EU) 2016/679. Any such processing must be grounded in a lawful basis, with explicit, informed, and specific consent from the data subject representing the most pertinent requirement for such secondary use. Consequently, the utilisation of an individual's personal data for model development without their unambiguous prior authorisation constitutes a violation of fundamental rights and the principles of data minimisation and purpose limitation enshrined in Union law.


In [16]:
relevant_docs = retriever.invoke(response)
blocks = format_docs_with_citations(relevant_docs)
print(blocks)

[SOURCE 1]
Reference : Chapter CHAPTER VI | Chapter CHAPTER VIMEASURES IN SUPPORT OF INNOVATION | Article Article 59sandbox
Page      : 92
---
(j) a short summary of the AI project developed in the sandbox, its objectives and expected results is published on the
website of the competent authorities; this obligation shall not cover sensitive operational data in relation to the activities
of law enforcement, border control, immigration or asylum authorities.
2.
For the purposes of the prevention, investigation, detection or prosecution of criminal offences or the execution of
criminal penalties, including safeguarding against and preventing threats to public security, under the control and
responsibility of law enforcement authorities, the processing of personal data in AI regulatory sandboxes shall be based on
a specific Union or national law and subject to the same cumulative conditions as referred to in paragraph 1.
3.
Paragraph 1 is without prejudice to Union or national law which ex

In [15]:
rag_chain.invoke(response)

'Based on the provided sources, the specific claim that processing personal data extracted from social media platforms for AI model training **without unambiguous prior consent** automatically constitutes a violation of fundamental rights and the principles of purpose limitation and data minimisation under Union law **is not explicitly confirmed or detailed**.\n\nThe sources establish a general framework where:\n1.  **GDPR applies:** The regulation explicitly states that the fundamental right to data protection is safeguarded by Regulations (EU) 2016/679 and others, and that principles like data minimisation and data protection by design apply when personal data is processed [SOURCE 8, SOURCE 7 (69)].\n2.  **High-risk AI systems have data governance requirements:** Providers of high-risk AI systems must ensure data sets are, to the best extent possible, complete and free of errors, and must pay specific attention to mitigating biases that could lead to discrimination [SOURCE 2 (68), SO